<h1 dir=rtl align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
    استارت‌آپ
</font>
</h1>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در این سوال می‌خواهیم مدلی طراحی کنیم تا بتواند پیش‌بینی گند یک استارت‌آپ می‌تواند موفق شود یا خیر.
</font>
</p>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در فایل اولیه‌ی این سوال دو فایل با نام‌های <code>train.csv</code> و <code>test.csv</code> قرار دارد که به ترتیب مجموعه‌داده‌ی آموزش و آزمون هستند.
    <br>
    مجموعه‌داده‌ی آموزش دارای ۵۷۶۱۵ سطر و ۱۲ ستون (ویژگی) است که توضیحات مربوط به این ستون‌ها در جدول زیر آمده است.
</font>
</p>

<center>
<div dir=rtl align=center style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>

| <b>نام ویژگی</b> | <b>توضیح ویژگی</b> |
| :---: | :---: |
| <code>name</code> | نام شرکت |
| <code>category_list</code> | زمینه‌ی کاری شرکت |
| <code>funding_total_usd</code> | مجموع بودجه‌ی شرکت (به دلار) |
| <code>status</code> | وضعیت شرکت (متغیر هدف مسئله که باید در ادامه کمی تغییرات در آن ایجاد کنید) |
| <code>country_code</code> | کد کشور |
| <code>state_code</code> | کد ایالت |
| <code>region</code> | منطقه |
| <code>city</code> | شهر |
| <code>funding_rounds</code> | تعداد دفعات تامین مالی شرکت |
| <code>founded_at</code> | تاریخ تاسیس |
| <code>first_funding_at</code> | تاریخ اولین تامین مالی شرکت |
| <code>last_funding_at</code> | تاریخ آخرین تامین مالی شرکت |

</font>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    مجموعه‌داده‌ی آزمون نیز دارای ۸۷۵۲ سطر است و ستون‌های آن مشابه مجموعه‌داده‌ی آموزش است، با این تفاوت که ستون <code>status</code> را ندارد.
</font>
</p>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    در ابتدا نیاز است تا کتابخانه‌های مورد نیاز خود را فراخوانی کنید. سپس با توجه به توضیحات بالا مجموعه‌داده‌های آموزش و آزمون را به نحو مناسبی بخوانید و پیش‌پردازش‌های لازم را روی آن‌ها انجام دهید.
    <br>
    اگر به داده‌ها دقت کنید مقادیر ستون <code>status</code> برابر با <code>operating</code> ،<code>closed</code> ،<code>acquired</code> و <code>ipo</code> است. 
    شرکتی را موفق می‌گوییم اگر در یکی از دو وضعیت <code>acquired</code> و <code>ipo</code> باشد. وضعیت <code>closed</code> به معنی این است که استارت‌آپ شکست خورده و شرکت بسته شده است و وضعیت <code>operating</code> نیز به این معنی است که شرکت به موفقیت نرسیده ولی هنوز ورشکست نشده است.
     بنابراین مدل شما در نهایت باید به عنوان پیش‌بینی یکی از سه عدد ۰ (شکست و بسته شده)، ۱ (عدم موفقیت ولی بشته نشده) و ۲ (موفقیت) را خروجی دهد.
</font>
</p>

In [74]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [10]:
train= pd.read_csv(r"train.csv")
test= pd.read_csv(r"test.csv")

In [88]:
df= train.copy()
X_test= test.copy()

df.replace('-', np.nan, inplace= True)
X_test.replace('-', np.nan, inplace= True)

df['status']= df['status'].apply(lambda x: 0 if (x=='closed') else 1 if (x=='operating') else 2)

df.drop(columns= ['name'], inplace= True)
X_test.drop(columns= ['name'], inplace= True)

df['category_list']= df['category_list'].apply(lambda x: list(map(str, x.lower().split('|'))) if (str(x)!='nan') else [])
X_test['category_list']= X_test['category_list'].apply(lambda x: list(map(str, x.lower().split('|'))) if (str(x)!='nan') else [])

df['funding_total_usd']= df['funding_total_usd'].astype(np.float64)
X_test['funding_total_usd']= df['funding_total_usd'].astype(np.float64)

df['first_funding_at']= pd.to_datetime(df['first_funding_at'], errors= 'coerce')
X_test['first_funding_at']= pd.to_datetime(X_test['first_funding_at'])

df['founded_at']= pd.to_datetime(df['founded_at'], errors= 'coerce')
X_test['founded_at']= pd.to_datetime(X_test['founded_at'])

df['last_funding_at']= pd.to_datetime(df['last_funding_at'], errors= 'coerce')
X_test['last_funding_at']= pd.to_datetime(X_test['last_funding_at'])

# df.dropna(subset= ['first_funding_at', 'status'], inplace= True)
# X_test.dropna(subset= ['first_funding_at'], inplace= True)

df['first_funding_days']= (df['first_funding_at'] - df['founded_at']).dt.days
df['last_funding_days']= (df['last_funding_at'] - df['founded_at']).dt.days
df['funding_days']= (df['last_funding_at'] - df['first_funding_at']).dt.days
df['founded_at_year']= df['founded_at'].dt.year
df['founded_at_month']= df['founded_at'].dt.month
df['founded_at_quarter']= df['founded_at'].dt.quarter
df['first_funding_at_year']= df['first_funding_at'].dt.year
df['first_funding_at_month']= df['first_funding_at'].dt.month
df['first_funding_at_quarter']= df['first_funding_at'].dt.quarter
df['last_funding_at_year']= df['last_funding_at'].dt.year
df['last_funding_at_month']= df['last_funding_at'].dt.month
df['last_funding_at_quarter']= df['last_funding_at'].dt.quarter
X_test['first_funding_days']= (X_test['first_funding_at'] - X_test['founded_at']).dt.days
X_test['last_funding_days']= (X_test['last_funding_at'] - X_test['founded_at']).dt.days
X_test['funding_days']= (X_test['last_funding_at'] - X_test['first_funding_at']).dt.days
X_test['founded_at_year']= X_test['founded_at'].dt.year
X_test['founded_at_month']= X_test['founded_at'].dt.month
X_test['founded_at_quarter']= X_test['founded_at'].dt.quarter
X_test['first_funding_at_year']= X_test['first_funding_at'].dt.year
X_test['first_funding_at_month']= X_test['first_funding_at'].dt.month
X_test['first_funding_at_quarter']= X_test['first_funding_at'].dt.quarter
X_test['last_funding_at_year']= X_test['last_funding_at'].dt.year
X_test['last_funding_at_month']= X_test['last_funding_at'].dt.month
X_test['last_funding_at_quarter']= X_test['last_funding_at'].dt.quarter
# df.drop(columns= ['founded_at', 'first_funding_at', 'last_funding_at'], inplace= True)
# X_test.drop(columns= ['founded_at', 'first_funding_at', 'last_funding_at'], inplace= True)

for col in ['category_list']:
    mlb= MultiLabelBinarizer()
    mocodes_encoded= mlb.fit_transform(df[col])
    mocodes_df= pd.DataFrame(mocodes_encoded, columns= [f'Mocode_{c}' for c in mlb.classes_], index= df.index)
    df= pd.concat([df.drop(columns= [col]).reset_index(drop= True), mocodes_df], axis= 1)
    mocodes_encoded= mlb.transform(X_test[col])
    mocodes_df= pd.DataFrame(mocodes_encoded, columns= [f'Mocode_{c}' for c in mlb.classes_], index= X_test.index)
    X_test= pd.concat([X_test.drop(columns= [col]).reset_index(drop= True), mocodes_df], axis= 1)

str_cols= df.select_dtypes(exclude= ['number']).columns.to_list()
encoder= OrdinalEncoder(handle_unknown= 'use_encoded_value', unknown_value= -1)
df[str_cols]= encoder.fit_transform(df[str_cols])
X_test[str_cols]= encoder.transform(X_test[str_cols])

df.dropna(subset= ['first_funding_at', 'status'], inplace= True)
X_test.dropna(subset= ['first_funding_at'], inplace= True)

df.drop(columns= ['founded_at', 'first_funding_at', 'last_funding_at'], inplace= True)
X_test.drop(columns= ['founded_at', 'first_funding_at', 'last_funding_at'], inplace= True)

X= df.drop(columns= ['status'])
y= df['status']

cat_cols= ['country_code', 'state_code', 'region', 'city', 'founded_at_year', 'founded_at_month', 'founded_at_quarter']
con_cols= ['funding_total_usd', 'first_funding_days', 'last_funding_days']
cat_imputer= SimpleImputer(strategy= 'most_frequent')
con_imputer= SimpleImputer(strategy= 'median')
preprocessor= ColumnTransformer(transformers= [
    ('cat_imputer', cat_imputer, cat_cols),
    ('con_imputer', con_imputer, con_cols)],
remainder= 'passthrough')
X= pd.DataFrame(preprocessor.fit_transform(X), columns= preprocessor.get_feature_names_out(), index= X.index)
X_test= pd.DataFrame(preprocessor.transform(X_test), columns= preprocessor.get_feature_names_out(), index= X_test.index)

# scaler= StandardScaler()
# X= pd.DataFrame(scaler.fit_transform(X), columns= X.columns, index= X.index)
# X_test= pd.DataFrame(scaler.transform(X_test), columns= X_test.columns, index= X_test.index)

smote= SMOTE(random_state= 48)
X_resample, y_resample= smote.fit_resample(X, y)

X_train, X_val, y_train, y_val= train_test_split(X_resample, y_resample, random_state= 48, test_size= 0.2, startify= y_resample)

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [87]:
y.isna().sum().sum()

0

In [79]:
y.isna().sum()

27

In [ ]:
X_train, X_val, y_train, y_val= train_test_split(X, y, random_state= 48, test_size= 0.2)

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
آموزش مدل
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    حال که داده را پاکسازی کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.
</font>
</p>

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معیار ارزیابی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> (با روش میانگین‌گیری <code>macro</code>) نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی مجموعه‌ی آموزش یا اعتبارسنجی ارزیابی کنید.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazir" size=3>
    برای دریافت نمره از این سوال لازم است تا دقت مدل شما از آستانه‌ی ۰.۴ بیشتر باشد.
    در صورتی که دقت مدل شما از ۰.۴ کمتر باشد نمره شما 
    <b>صفر</b>
    خواهد شد و در غیر این صورت با فرمول زیر محاسبه می‌شود:
</font>
</p>


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score
import warnings

In [6]:
warnings.filterwarnings('ignore')

In [7]:
lr_model= LogisticRegression(random_state= 48)
lr_model.fit(X_train, y_train)
predict= lr_model.predict(X_val)
f1_score(y_val, predict, average= 'macro')

0.4258245320065521

In [8]:
rf_model= RandomForestClassifier(random_state= 48)
param_grid= {
    'n_estimators': [10, 100, 1000],
    'max_depth': [5, 10, 100]
    }
grid= GridSearchCV(rf_model, param_grid, cv= 4, scoring= 'r2', n_jobs= -1, verbose= 2)
grid.fit(X_train, y_train)
print(grid.best_params_)
predict= grid.best_estimator_.predict(X_val)
f1_score(y_val, predict, average= 'macro')

Fitting 4 folds for each of 9 candidates, totalling 36 fits
{'max_depth': 10, 'n_estimators': 10}


0.29897946637157263

In [ ]:
xgb_model= XGBClassifier(random_state= 48)
param_grid= {
    'n_estimators': [10, 100, 1000],
    'max_depth': [5, 10, 100],
    'learning_rate': [0.01, 0.1]
}
grid= GridSearchCV(xgb_model, param_grid, cv= 4, scoring= 'r2', n_jobs=-1, verbose= 2)
grid.fit(X_train.to_numpy(), y_train)
print(grid.best_params_)
predict= grid.best_estimator_.predict(X_val.to_numpy())
f1_score(y_val, predict, average= 'macro')

Fitting 4 folds for each of 18 candidates, totalling 72 fits


: 

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 پیش‌بینی بر روی داده‌ی تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    پیش‌بینی مدل خود بر روی داده‌های آزمون را در یک دیتافریم (<code>dataframe</code>) به فرمت زیر ذخیره کنید.
</font>
</p>


<p dir=rtl style="direction: rtl;text-align: justify;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    توجه داشته باشید که نام دیتافریم باید <code>submission</code> باشد؛ در غیر این‌صورت، سامانه‌ی داوری قادر به ارزیابی خروجی شما نخواهد بود.
    این دیتافریم صرفا شامل یک ستون به نام <code>status</code> است و ۸۷۵۲ سطر دارد.
    <br>
    به ازای هر سطر موجود در مجموعه‌داده‌ی آزمون، باید یک مقدار پیش‌بینی‌شده داشته باشید که مقدار <code>status</code> پیش‌بینی مدل شما است.
    به‌عنوان مثال جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته این مقادیر به‌صورت فرضی هستند و در جواب شما، ممکن است متفاوت باشند.
</font>
</p>

<center>
<div align=center 
style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
||<code>status</code>|
|:----:|:-----:|
|0|1|
|1|2|
|2|1|
|3|1|
|4|0|

</font>
</div>
</center>

In [ ]:
# To-Do
submission =

<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) در غیر این صورت، در پایان مسابقه نمره شما به صفر تغییر خواهد کرد.
    <br>
    همچنین اگر از کولب برای اجرای این فایل نوت‌بوک استفاده می‌کنید، قبل از ارسال فایل <code>result.zip</code>، آخرین نسخه‌ی نوت‌بوک خود را دانلود کرده و داخل فایل ارسالی قرار دهید.
</font>

In [ ]:
import zipfile

if not os.path.exists(os.path.join(os.getcwd(), 'startup.ipynb')):
    %notebook -e startup.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)

file_names = ['startup.ipynb', 'submission.csv']
compress(file_names)